In [1]:
import numpy as np
import cv2
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Model
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [2]:
img = cv2.imread("mountain.jpg")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)      
img = cv2.resize(img, (224, 224)) 
img = np.expand_dims(img, axis=0) 

In [3]:
print(img.shape)

(1, 224, 224, 3)


In [4]:
processed_img = preprocess_input(img)

In [5]:
base_model = VGG16(weights='imagenet', include_top=False)
model = Model(inputs=base_model.input, outputs=base_model.get_layer('block3_conv3').output)

In [6]:
features = model.predict(img)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 630ms/step


In [7]:
features.shape

(1, 56, 56, 256)

In [8]:
features = features.reshape(-1,features.shape[-1])

In [9]:
features.shape

(3136, 256)

In [10]:
from sklearn.decomposition import PCA

In [11]:
pca = PCA(n_components=3)
rgb = pca.fit_transform(features)

In [12]:
print(rgb.min(),rgb.max())

-7673.4937 13303.996


In [13]:
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())

In [14]:
print(rgb.min(),rgb.max())

0.0 1.0


In [15]:
rgb = (rgb * 255).astype(np.uint8)

In [16]:
print(rgb.min(),rgb.max())

0 255


In [17]:
rgb.shape

(3136, 3)

In [18]:
base_model.trainable = False

In [19]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None, None, 3)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, None, None, 64) │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, None, None, 64) │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, None, None, 64) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, None, None,     │        73,856 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, None, None,     │       147,584 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, None, None,     │             0 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, None, None,     │       295,168 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, None, None,     │       590,080 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, None, None,     │       590,080 │
│                                 │ 256)                   │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,735,488 (6.62 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 1,735,488 (6.62 MB)

In [ ]:
model = KMeans(n_clusters=5,init='k-means++')
model.fit(rgb)
centroids = model.cluster_centers_
colors = np.array(centroids, dtype='uint8')

cnt = 0
for color in colors:
        plt.axis('off')
        mat = np.zeros((3, 3, 3), dtype='uint8')
        mat[:, :, :] = color

        plt.imshow(mat)
        plt.show()